# 🧬 19 — How 3D models see crowded tomograms

**Audience:** biology student moving toward research · **Time:** ~4 hours · **Hardware:** free Colab CPU · **Framework:** NumPy · **Prerequisites:** 09–10, 13–15; convolution basics

**Goals:** explain the concept, run and modify the toy model, identify one failure mode, and connect the method to an original paper. All data are synthetic and for research and education.


## 1. Biological and computational motivation

A 3D network analyzes small neighborhoods throughout a tomogram. Local filters can recognize edges and textures; deeper layers assemble wider context. Different research papers add dilated filters, multiple views, graph relations, attention, or topology constraints. Architecture is only one part of the experiment: dense labels, class balance, patch sampling, memory, and independent test specimens often decide whether the result generalizes.


## 2. Theory and assumptions

A 3×3×3 convolution has 27 spatial weights per input-output channel pair. Two stride-one layers have a nominal 5×5×5 receptive field. Dilation expands reach without sampling every intermediate voxel. For a float32 `256³` volume, one feature map uses about 64 MiB; 64 such maps use about 4 GiB before gradients and optimizer state. This motivates patching, downsampling, and careful memory accounting. Segmentation quality needs overlap, boundaries, topology, and downstream biological checks.


## 3. Runnable experiment

Predict a result before running. In Colab, NumPy is included; elsewhere install it with `%pip install numpy` if needed.


In [ ]:
import numpy as np
SEED=7
print("NumPy",np.__version__,"| device: CPU | seed:",SEED)


In [ ]:
volume=np.zeros((9,9,9)); volume[4,4,4]=1
kernel=np.ones((3,3,3))/27
filtered=np.zeros((7,7,7))
for z in range(7):
    for y in range(7):
        for x in range(7):
            filtered[z,y,x]=np.sum(volume[z:z+3,y:y+3,x:x+3]*kernel)
size_mib=256**3*4/2**20
print('nonzero voxels after local filter:',np.count_nonzero(filtered))
print('one 256³ float32 map:',size_mib,'MiB; 64 maps:',size_mib*64/1024,'GiB')
assert np.count_nonzero(filtered)==27


## 4. Interpretation and challenge

The code is a fixed linear filter, not a learned neural network. It illustrates local context and memory scale. A high Dice score alone can miss biologically important breaks in a membrane or merges between adjacent particles.

**Change one thing:** Draw the receptive field after three 3×3×3 stride-one layers. Then sketch a tomogram-level split that prevents neighboring crops from leaking into both training and test sets.


## 5. Paper bridge

Compare [3D Dilated Dense Network](https://doi.org/10.1109/TCBB.2021.3065986), [Vox-UDA](https://xulabs.github.io/publication/), [MVGFormer](https://docs.google.com/document/d/e/2PACX-1vQOw1_7_1a6HuyyvcwmtOzfliOmyz4L7giYkp3-i0MOxh-ASU0ZCjNtdYZZ4KC3UlFHdC9Gb-gzXMsi/pub), and [TopoFuse](https://xulabs.github.io/publication/).

Record the paper’s input, output, supervision, split unit, central assumption, strongest comparison, and one alternate explanation.
